MD# Bônus: um qubit no seu Colab*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/souuzaa/gdg-blumenau-ml-gcp/blob/main/05-bonus-quantico/qubits.ipynb)

\* simulado. Mas o código é o mesmo que roda num computador quântico de verdade, trocando só o *device*.

In [ ]:
%pip install -q pennylane cirq-core

MD## 1. Superposição

Uma porta Hadamard tira o qubit do 0 e coloca em 50/50.

In [ ]:
import pennylane as qml

dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def superposicao():
    qml.Hadamard(wires=0)
    return qml.probs(wires=0)

superposicao()  # [0.5 0.5]

MD## 2. Interferência

Duas Hadamard seguidas: as amplitudes de 1 se cancelam e o qubit volta para 0. Superposição é o cenário, interferência é o truque.

In [ ]:
@qml.qnode(dev)
def interferencia():
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=0)
    return qml.probs(wires=0)

interferencia()  # [1. 0.]

MD## 3. Emaranhamento

Hadamard + CNOT. Só saem 00 ou 11. Mediu um, sabe o outro.

In [ ]:
dev2 = qml.device("default.qubit", wires=2)

@qml.qnode(dev2)
def bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

print(qml.draw(bell)())
bell()  # [0.5 0.  0.  0.5]

MD## 4. Um circuito que aprende

Os ângulos das portas são pesos treináveis. O loop é igual ao de machine learning clássico: forward, custo, gradiente, atualiza.

In [ ]:
from pennylane import numpy as np

@qml.qnode(dev2)
def circuito(params):
    qml.RX(params[0], wires=0)
    qml.RY(params[1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RZ(params[2], wires=0)
    qml.RY(params[3], wires=1)
    return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))

params = np.array([0.1, 0.2, 0.3, 0.4], requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.35)
historico = []
for passo in range(60):
    params, custo = opt.step_and_cost(circuito, params)
    historico.append(float(custo))
print(f"custo final: {float(circuito(params)):+.4f} (mínimo teórico -1)")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(historico)
plt.xlabel("passo")
plt.ylabel("custo")
plt.title("Gradiente descendente num circuito quântico")
plt.show()

MD## 5. O mesmo Bell em Cirq, a biblioteca do Google Quantum AI

Agora medindo de verdade: 1.000 execuções e a contagem de cada resultado.

In [ ]:
import cirq

q0, q1 = cirq.LineQubit.range(2)
circuito_bell = cirq.Circuit(cirq.H(q0), cirq.CNOT(q0, q1), cirq.measure(q0, q1, key="m"))
print(circuito_bell)

resultado = cirq.Simulator(seed=42).run(circuito_bell, repetitions=1000)
contagem = resultado.histogram(key="m")
{format(k, "02b"): contagem.get(k, 0) for k in range(4)}

MD## Para ir além

- Experimento completo da talk (soja, milho e câmbio; Monte Carlo contra Quantum Amplitude Estimation): [souuzaa/quantum-backtest-sample](https://github.com/souuzaa/quantum-backtest-sample)
- [Tutoriais do Cirq](https://quantumai.google/cirq/start/basics)
- *Quantum Computing for the Quantum Curious* (Springer, acesso aberto)